# Prompt 02: Anatomy of a Prompt

1. Build a prompt layer by layer; measure each addition.
2. Demonstrate few-shot as message-list vs inline.
3. Assistant prefix trick for reliable JSON.
4. Demonstrate order sensitivity (task at top vs bottom).
5. Exercises: template your own feature's prompt.

Deps: `pip install anthropic openai`

In [ ]:
import os

def call_messages(messages, max_tokens=400, system=None):
    if os.getenv('ANTHROPIC_API_KEY'):
        from anthropic import Anthropic
        kw = dict(model='claude-sonnet-4-6', max_tokens=max_tokens, temperature=0, messages=messages)
        if system: kw['system'] = system
        r = Anthropic().messages.create(**kw)
        return r.content[0].text
    if os.getenv('OPENAI_API_KEY'):
        from openai import OpenAI
        msgs = ([{'role':'system','content':system}] if system else []) + messages
        r = OpenAI().chat.completions.create(model='gpt-4o-mini', max_tokens=max_tokens, temperature=0, messages=msgs)
        return r.choices[0].message.content
    return '[no provider]'

## 1. Add One Layer at a Time

In [ ]:
input_text = 'Alice joined NovaCore on 2022-03-01 as a senior engineer reporting to Bob.'

layers = [
    # Bare
    {'system': None,
     'messages': [{'role':'user','content':'Extract who, role, date. ' + input_text}]},
    # + system identity
    {'system': 'You are a precise extraction system.',
     'messages': [{'role':'user','content':'Extract who, role, date. ' + input_text}]},
    # + format spec
    {'system': 'You are a precise extraction system. Return JSON only: {"name":str,"role":str,"date":"YYYY-MM-DD"}.',
     'messages': [{'role':'user','content':'Extract who, role, date. ' + input_text}]},
    # + delimited input
    {'system': 'You are a precise extraction system. Return JSON only: {"name":str,"role":str,"date":"YYYY-MM-DD"}.',
     'messages': [{'role':'user','content':'Extract who, role, date.\n<text>' + input_text + '</text>\nReturn JSON.'}]},
    # + few-shot
    {'system': 'You are a precise extraction system. Return JSON only: {"name":str,"role":str,"date":"YYYY-MM-DD"}.',
     'messages': [
         {'role':'user','content':'<text>Carol joined Acme on 2020-07-15 as a director.</text>'},
         {'role':'assistant','content':'{"name":"Carol","role":"director","date":"2020-07-15"}'},
         {'role':'user','content':'<text>Dave joined Beta on 2019-11-02 as a principal engineer.</text>'},
         {'role':'assistant','content':'{"name":"Dave","role":"principal engineer","date":"2019-11-02"}'},
         {'role':'user','content':'<text>' + input_text + '</text>'},
     ]},
]

for i, cfg in enumerate(layers):
    print(f'--- layer {i} ---')
    print(call_messages(cfg['messages'], system=cfg['system']))
    print()

## 2. Few-Shot: Message List vs Inline

In [ ]:
inline = '''Extract to JSON: {"name":str,"role":str,"date":"YYYY-MM-DD"}.

Input: Carol joined Acme on 2020-07-15 as a director.
Output: {"name":"Carol","role":"director","date":"2020-07-15"}

Input: Dave joined Beta on 2019-11-02 as a principal engineer.
Output: {"name":"Dave","role":"principal engineer","date":"2019-11-02"}

Input: ''' + input_text + '''
Output:'''

print('--- inline few-shot ---')
print(call_messages([{'role':'user','content':inline}]))

print('\n--- message-list few-shot ---')
msgs = [
    {'role':'user','content':'Extract to JSON: <text>Carol joined Acme on 2020-07-15 as a director.</text>'},
    {'role':'assistant','content':'{"name":"Carol","role":"director","date":"2020-07-15"}'},
    {'role':'user','content':'Extract to JSON: <text>Dave joined Beta on 2019-11-02 as a principal engineer.</text>'},
    {'role':'assistant','content':'{"name":"Dave","role":"principal engineer","date":"2019-11-02"}'},
    {'role':'user','content':'Extract to JSON: <text>' + input_text + '</text>'},
]
print(call_messages(msgs, system='Return JSON only.'))

## 3. Assistant Prefix Trick (Claude)

Seeding the assistant with `{"` almost eliminates JSON prefix errors.

In [ ]:
if os.getenv('ANTHROPIC_API_KEY'):
    from anthropic import Anthropic
    client = Anthropic()

    messages = [
        {'role':'user','content':'Return a JSON object with name, role, date for: ' + input_text},
        {'role':'assistant','content':'{'},   # prefill
    ]
    r = client.messages.create(
        model='claude-sonnet-4-6', max_tokens=200, temperature=0,
        system='Return JSON only.',
        messages=messages,
    )
    # Assistant prefix is NOT included in the response; prepend it when parsing.
    body = '{' + r.content[0].text
    print('raw:', body)
    import json
    print('parsed:', json.loads(body[:body.rfind('}')+1]))
else:
    print('Skipped (needs ANTHROPIC_API_KEY).')

## 4. Order Sensitivity: Task at Top vs Bottom

In [ ]:
big_context = 'Lorem ipsum filler. ' * 800   # ~1500 tokens of distracting content

task_top = f'Return the CEO name mentioned in the document. Just the name.\n\n{big_context}\n<doc>NovaCore CEO Maya Kapoor announced today that...</doc>'
task_bottom = f'{big_context}\n<doc>NovaCore CEO Maya Kapoor announced today that...</doc>\n\nReturn the CEO name mentioned in the document. Just the name.'

print('--- task-at-top ---')
print(call_messages([{'role':'user','content':task_top}], max_tokens=80))
print('\n--- task-at-bottom ---')
print(call_messages([{'role':'user','content':task_bottom}], max_tokens=80))

## 5. Exercise: Template Your Own Feature

Pick a real feature (e.g., classify-support-tickets, summarize-meeting-notes). Produce:

- SYSTEM (identity + scope + format + constraints + tools)
- CONTEXT block (if any)
- 3 few-shot pairs
- TASK template

Run it on 10 inputs. Where does it fail? Iterate. The skeleton doesn't change much; what changes is your eval loop. That's in session 10.

## 6. Takeaways

- **Prompts have four layers:** system, context, few-shot, task.
- **Message-list few-shot > inline few-shot** for reliability.
- **Assistant prefix** is a cheat code for JSON shape.
- **Put the task at the end** when context is long — lost-in-middle is real.
- **Delimit user-supplied content** (XML tags, fences) for clarity and safety.

Next: **03 — Zero-Shot, Few-Shot, N-Shot** — when examples help, how many, and how to choose them.